# Training Pipeline

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [3]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

## Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:,0], test_size=0.2)

## Scaling

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

## Encoding

In [7]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.fit_transform(y_test)

## Numpy Arrays to Torch

In [8]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

## Defining the Model

In [9]:
class MySimpleNN:
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss_function(self,y_pred, y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        loss = -(y_train_tensor*torch.log(y_pred) + (1-y_train_tensor)*torch.log(1-y_pred)).mean()
        return loss

In [10]:
learning_rate= 0.1
epochs = 1000

## Training Pipeline

In [11]:
#create model
model = MySimpleNN(X_train_tensor)

#defineloop
for epoch in range(epochs):
    #forwardpass
    y_pred = model.forward(X_train_tensor)

    #losscalculate
    loss = model.loss_function(y_pred, y_train_tensor)
    
    #backwardpass
    loss.backward()

    #parametersupdate
    with torch.no_grad():
        model.weights -= learning_rate*model.weights.grad
        model.bias -= learning_rate * model.bias.grad
    
    #zerogradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    print (f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 4.107949047026968
Epoch: 2, Loss: 4.0064128894741895
Epoch: 3, Loss: 3.9010538386045823
Epoch: 4, Loss: 3.7896129203387483
Epoch: 5, Loss: 3.6692010433452786
Epoch: 6, Loss: 3.5473805647041523
Epoch: 7, Loss: 3.4207394419012793
Epoch: 8, Loss: 3.2863395138280156
Epoch: 9, Loss: 3.147917555008325
Epoch: 10, Loss: 3.0059778718985566
Epoch: 11, Loss: 2.8550114850407478
Epoch: 12, Loss: 2.7027030188598014
Epoch: 13, Loss: 2.5508394901838796
Epoch: 14, Loss: 2.395810321250701
Epoch: 15, Loss: 2.2384391020229772
Epoch: 16, Loss: 2.079435128625615
Epoch: 17, Loss: 1.9237182191825246
Epoch: 18, Loss: 1.7759849036652866
Epoch: 19, Loss: 1.6372625896464796
Epoch: 20, Loss: 1.5073267586907864
Epoch: 21, Loss: 1.3847013774628107
Epoch: 22, Loss: 1.2724034574458265
Epoch: 23, Loss: 1.17470991343878
Epoch: 24, Loss: 1.0923028779559723
Epoch: 25, Loss: 1.024912431862263
Epoch: 26, Loss: 0.9712845330694161
Epoch: 27, Loss: 0.9294587928999649
Epoch: 28, Loss: 0.8971242189142883
Epoch: 2

In [12]:
model.weights

tensor([[-2.6031e-01],
        [-4.7975e-03],
        [ 2.4870e-01],
        [-2.6362e-01],
        [ 5.4230e-02],
        [-1.2112e-03],
        [ 3.2890e-02],
        [ 8.2392e-05],
        [ 3.0434e-02],
        [-1.1054e-01],
        [-2.0932e-01],
        [-9.5218e-03],
        [ 1.3146e-01],
        [ 5.6726e-02],
        [ 1.7646e-02],
        [ 6.7743e-02],
        [ 3.3359e-02],
        [-8.7761e-02],
        [ 3.9496e-02],
        [-2.0609e-02],
        [ 3.1537e-02],
        [ 4.6434e-03],
        [ 1.9204e-01],
        [ 3.3164e-02],
        [-4.6427e-02],
        [-1.8269e-01],
        [-3.1974e-02],
        [ 1.1464e-01],
        [-6.3321e-02],
        [ 1.3976e-01]], dtype=torch.float64, requires_grad=True)

In [13]:
model.bias

tensor([-0.5839], dtype=torch.float64, requires_grad=True)

In [14]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()

print(y_pred)

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
      

In [15]:
accuracy = (y_pred == y_test_tensor).float().mean()
print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5701754093170166
